In [ ]:
df = pd.read_excel("/content/Copy of dv_dataset_with_all_program_info.xlsx",sheet_name="dv_dataset_with_all_program_inf")


In [ ]:
import pandas as pd
import numpy as np
from statsmodels.formula.api import ols
#This analysis prepares data by filling missing values, log-transforming the DV, standardizing categorical variables, setting "None" as the reference group, and estimating an OLS regression with Brand as a moderator to test whether program effects vary across different brands.
# ========= 0. Key fix: First fix None =========
df["program_aim_a_b"] = df["program_aim_a_b"].fillna("None")

# ========= 1. Select the variable =========
need_cols = ["cum_sales_m12", "program_aim_a_b", "brand"]
df_brand = df[need_cols].dropna(subset=["cum_sales_m12", "brand"]).copy()
# Note: program dropna is no longer referred to here

# ========= 2. Log-transform the DV =========
df_brand["log_cum_sales_m12"] = np.log1p(df_brand["cum_sales_m12"])

# ========= 3. Unify program names (without merging categories) =========
df_brand["program_clean"] = (
    df_brand["program_aim_a_b"]
    .astype(str)
    .str.strip()
)

# （Only replace if there are spelling aliases; skip if not needed）
df_brand["program_clean"] = df_brand["program_clean"].replace({
    "AIM-A": "AIM_A",
    "AIM-B": "AIM_B",
    "OneSource": "One Source",
})

# ========= 4. Clearly define 7 categories (None as the benchmark) =========
program_levels = [
    "None",
    "AIM_A",
    "AIM_B",
    "AIM_A & One Source",
    "AIM_B & One Source",
    "AIM_Both",
    "One Source"
]

df_brand = df_brand[df_brand["program_clean"].isin(program_levels)].copy()

df_brand["program_clean"] = pd.Categorical(
    df_brand["program_clean"],
    categories=program_levels,
    ordered=False
)

df_brand["brand"] = df_brand["brand"].astype("category")

# ========= 5. Check =========
print("Program Distribution (7 categories)")
print(df_brand["program_clean"].value_counts(dropna=False))

# ========= 6. Regression：Brand × Program =========
formula = """
log_cum_sales_m12
~ C(program_clean) * C(brand)
"""

model = ols(formula, data=df_brand).fit()
print(model.summary())


In [ ]:
# ========= 1.Make sure to use the df_brand that you just cleaned up =========
# It should contain:
# log_cum_sales_m12, brand, program_clean

# ========= 2. Calculate the mean =========
plot_df = (
    df_brand
    .groupby(["brand", "program_clean"], observed=True)["log_cum_sales_m12"]
    .mean()
    .reset_index()
)

# ========= 3. Draw a folded line graph =========
plt.figure(figsize=(12, 6))

for brand in plot_df["brand"].unique():
    sub = plot_df[plot_df["brand"] == brand]
    plt.plot(
        sub["program_clean"],
        sub["log_cum_sales_m12"],
        marker="o",
        label=brand
    )

plt.xlabel("Program")
plt.ylabel("Average log(Cumulative Sales 12M)")
plt.title("Average 12-Month Log Sales by Program and Brand")
plt.legend(title="Brand", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=30)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from statsmodels.formula.api import ols

print("="*60)
print("Division（Region）Analysis (No fixed effect)")
print("DV: log(cum_sales_m12)")
print("Baseline: program = None")
print("="*60)

# =====================================================
# 0. Key Step: Keep None (very important)
# =====================================================
df["program_aim_a_b"] = df["program_aim_a_b"].fillna("None")

# =====================================================
# 1. Select variables & Clean data
# =====================================================
need_cols = [
    "cum_sales_m12",
    "program_aim_a_b",
    "division_code"
]

df_div = df[need_cols].dropna(subset=["cum_sales_m12", "division_code"]).copy()
# do not program dropna (None has been explicitly retained)

# =====================================================
# 2. Log-transform the DV
# =====================================================
df_div["log_cum_sales_m12"] = np.log1p(df_div["cum_sales_m12"])

# =====================================================
# 3. Program： Retain 7 categories (do not merge
# =====================================================
df_div["program_clean"] = (
    df_div["program_aim_a_b"]
    .astype(str)
    .str.strip()
)

# （Only replace if there are spelling aliases; skip if not needed）
df_div["program_clean"] = df_div["program_clean"].replace({
    "AIM-A": "AIM_A",
    "AIM-B": "AIM_B",
    "OneSource": "One Source",
})

program_levels = [
    "None",
    "AIM_A",
    "AIM_B",
    "AIM_A & One Source",
    "AIM_B & One Source",
    "AIM_Both",
    "One Source"
]

df_div = df_div[df_div["program_clean"].isin(program_levels)].copy()

df_div["program_clean"] = pd.Categorical(
    df_div["program_clean"],
    categories=program_levels,
    ordered=False
)

# =====================================================
# 4. Set "Division" as the categorical variable
# =====================================================
df_div["division_code"] = df_div["division_code"].astype("category")

print("✓ Program  Distribution (7 Categories）")
print(df_div["program_clean"].value_counts(dropna=False))
print(f"\nN={len(df_div)} | divisions={df_div['division_code'].nunique()}")

# =====================================================
# 5. Regression：Division × Program（"None" as the benchmark）
# =====================================================
formula = """
log_cum_sales_m12
~ C(program_clean) * C(division_code)
"""

model_div = ols(formula, data=df_div).fit()

# =====================================================
# 6. Output result
# =====================================================
print("\n" + "="*60)
print("Output result（Division × Program，DV=log）")
print("="*60)
print(model_div.summary())


In [ ]:
plot_div = (
    df_div
    .groupby(["division_code", "program_clean"], observed=True)["log_cum_sales_m12"]
    .mean()
    .reset_index()
)



top5_divisions = (
    plot_div
    .groupby("division_code")["log_cum_sales_m12"]
    .mean()
    .sort_values(ascending=False)
    .head(5)
    .index
)

other_divisions = (
    plot_div["division_code"]
    .unique()
)

other_divisions = [d for d in other_divisions if d not in top5_divisions]

print("Top 5 divisions:", list(top5_divisions))
print("Other divisions:", other_divisions)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Select Top 5 Divisions (based on overall average sales volume)
top5_divisions = (
    plot_div
    .groupby("division_code")["log_cum_sales_m12"]
    .mean()
    .sort_values(ascending=False)
    .head(5)
    .index
)

plot_top5 = plot_div[plot_div["division_code"].isin(top5_divisions)].copy()

programs = plot_top5["program_clean"].cat.categories
x = np.arange(len(programs))
bar_width = 0.15

plt.figure(figsize=(12, 6))

for i, div in enumerate(top5_divisions):
    sub = plot_top5[plot_top5["division_code"] == div]
    sub = sub.set_index("program_clean").reindex(programs)

    plt.bar(
        x + i * bar_width,
        sub["log_cum_sales_m12"],
        width=bar_width,
        label=f"Division {div}"
    )

plt.xlabel("Program")
plt.ylabel("Average log(Cumulative Sales 12M)")
plt.title("Average 12-Month Log Sales by Program\nTop 5 Divisions")
plt.xticks(x + bar_width * (len(top5_divisions) - 1) / 2, programs, rotation=30)
plt.legend(
    title="Division",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)
plt.tight_layout(rect=[0, 0, 0.85, 1])



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
# Select other 4 Divisions

plot_others = plot_div[plot_div["division_code"].isin(other_divisions)].copy()

programs = plot_others["program_clean"].cat.categories
x = np.arange(len(programs))
bar_width = 0.18
plt.figure(figsize=(12, 6))

for i, div in enumerate(other_divisions):
    sub = plot_others[plot_others["division_code"] == div]
    sub = sub.set_index("program_clean").reindex(programs)

    plt.bar(
        x + i * bar_width,
        sub["log_cum_sales_m12"],
        width=bar_width,
        label=f"Division {div}"
    )

plt.xlabel("Program")
plt.ylabel("Average log(Cumulative Sales 12M)")
plt.title("Average 12-Month Log Sales by Program\nOther Divisions")
plt.xticks(
    x + bar_width * (len(other_divisions) - 1) / 2,
    programs,
    rotation=30
)

plt.legend(
    title="Division",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.grid(axis="y", alpha=0.3)
plt.tight_layout(rect=[0, 0, 0.85, 1])
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# =====================================================
# 0. Preserve "None" as an explicit category
# =====================================================
df["program_aim_a_b"] = df["program_aim_a_b"].fillna("None")

# =====================================================
# 1. Select required columns and clean data
# =====================================================
need_cols = ["cum_sales_m12", "division_code", "program_aim_a_b"]
df_clean = df[need_cols].dropna(subset=["cum_sales_m12", "division_code"]).copy()

# Log-transform the dependent variable
df_clean["log_cum_sales_m12"] = np.log1p(df_clean["cum_sales_m12"])

# =====================================================
# 2. Program variable: keep all 7 categories (no merging)
# =====================================================
df_clean["program_clean"] = (
    df_clean["program_aim_a_b"]
    .astype(str)
    .str.strip()
).replace({
    "AIM-A": "AIM_A",
    "AIM-B": "AIM_B",
    "OneSource": "One Source",
})

program_levels = [
    "None",
    "AIM_A",
    "AIM_B",
    "AIM_A & One Source",
    "AIM_B & One Source",
    "AIM_Both",
    "One Source"
]

# Keep only valid program categories
df_clean = df_clean[df_clean["program_clean"].isin(program_levels)].copy()

df_clean["program_clean"] = pd.Categorical(
    df_clean["program_clean"],
    categories=program_levels,
    ordered=False
)

# =====================================================
# 3. Create heatmap matrix: average log sales
# =====================================================
heatmap_avg = (
    df_clean
    .groupby(["division_code", "program_clean"])["log_cum_sales_m12"]
    .mean()
    .unstack()
)

print(heatmap_avg.head())

# =====================================================
# 4. Order divisions by overall average log sales
# =====================================================
region_order = (
    df_clean
    .groupby("division_code")["log_cum_sales_m12"]
    .mean()
    .sort_values()
    .index
)

heatmap_avg = heatmap_avg.reindex(region_order)

# =====================================================
# 5. Swap the 3rd and 4th columns (index-based)
# =====================================================
cols = list(heatmap_avg.columns)

# Swap positions: index 2 <-> index 3
cols[2], cols[3] = cols[3], cols[2]

heatmap_avg = heatmap_avg[cols]

print("Final column order:", heatmap_avg.columns.tolist())

# =====================================================
# 6. Color scale settings
# =====================================================
vals = heatmap_avg.stack().dropna()
center_val = vals.median()
vmin = vals.quantile(0.10)
vmax = vals.quantile(0.90)

print("Color scale:")
print("Center =", center_val)
print("vmin, vmax =", vmin, vmax)

custom_cmap = LinearSegmentedColormap.from_list(
    "blue_purple_red",
    ["#2166ac", "#7b3294", "#b2182b"]
)

# =====================================================
# 7. Plot heatmap
# =====================================================
plt.figure(figsize=(10, 8))

sns.heatmap(
    heatmap_avg,
    cmap=custom_cmap,
    center=center_val,
    vmin=vmin,
    vmax=vmax,
    linewidths=0.4,
    linecolor="white",
    cbar_kws={"label": "Average Log Sales"}
)

plt.title("Heatmap (Average): Division × Program Participation")
plt.xlabel("Program Participation")
plt.ylabel("Division")
plt.tight_layout()
plt.show()
